In [ ]:
# All required dependencies for this notebook loaded here

from openai import OpenAI
import os
from pdf2image import convert_from_path
from pathlib import Path
from urllib.request import urlretrieve
from pathlib import Path
import base64
import json

In [ ]:
# Replace *** with your OpenAI API Key

client = OpenAI(api_key = "***")

In [ ]:
# Download an example PDF from FDA
url = "https://web.archive.org/web/20170210061844im_/http://www.fda.gov/ohrms/dockets/ac/99/transcpt/3505t1.pdf"

output_file = "example.pdf"

urlretrieve(url, output_file)

In [ ]:
# Specify PDF here
pdf = Path("example.pdf")

# Specify Output Folder for images and create it
out = Path("example")
out.mkdir(exist_ok=True)

images = convert_from_path(pdf, output_folder=out, fmt="png", output_file=pdf.stem)

In [ ]:
# Encode images for processing

# Small helper function to encode
def encode_image(image_path):
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

image_path = Path("example/example0001-061.png")

encoded_image = encode_image(image_path=image_path)

In [ ]:
# Process image and save as text

prompt = f"""You are a professional OCR engine Transcribe the text EXACTLY as written.
Do not correct spelling or grammar errors.
If text is blurry use context to infer the word.
Ignore headers footers as well as line and page numbers.
"""

response = client.responses.create(
    model="gpt-4.1-mini-2025-04-14",
    input=[
        {
            "role": "user",
            "content": [
                { "type": "input_text", "text": prompt},
                {
                    "type": "input_image",
                    "image_url": f"data:image/jpeg;base64,{encoded_image}",
                },
            ],
        }
    ],
)

In [ ]:
# Write the code to text file for further processing

text = response.output_text

with open("example/example0001-061.txt", "w", encoding="utf-8") as f:
    f.write(response.output_text)

In [ ]:
# Read Text file and process it

prompt = f"""
You are an expert transcript cleaner. You will receive a raw text chunk. Your goal is to extract the dialogue into a structured format.

**PRIME DIRECTIVE - WORD FIDELITY VS. FORMAT FLEXIBILITY:**
1. **DO NOT** change words, fix grammar,  remove non-verbal markers (e.g., [Laughter.], (Applause.)) or autocomplete sentences.
2. **DO** fix whitespace. You must remove line breaks within a sentence and normalize multiple spaces into a single space.

Follow these processing rules:

1. **Scope & Exclusions:**
   - **Ignore Metadata:** Do not extract meeting rosters, attendee lists, or headers. Start extracting only when the actual dialogue begins.
   - **Ignore Artifacts:** Remove page numbers, file paths, and margin line numbers.

2. **Speaker Identification:**
   - **Standard Speech:** Extract the speaker's name exactly as written and convert to UPPERCASE.
   - **Orphaned Speech:** If the chunk starts with sentences/dialogue but has no speaker name attached, label the speaker as: UNKNOWN.
"""

structured_output = {
    "type": "json_schema",
    "json_schema": {
        "name": "transcript_response",
        "schema": {
            "type": "object",
            "properties": {
                "entries": {
                    "type": "array",
                    "items": {
                        "type": "object",
                        "properties": {
                            "speaker": {
                                "type": "string",
                                "description": "The speaker's name in UPPERCASE. Use 'UNKNOWN' if the text starts without a name. Do not extract names from headers or rosters."
                            },
                            "statement": {
                                "type": "string",
                                "description": "The spoken text verbatim, including all bracketed [...] or parenthetical (...) notes like [Laughter.], (Applause.). Remove all line breaks (\\n) so it is a single line."
                            }
                        },
                        "required": ["speaker", "statement"],
                        "additionalProperties": False
                    }
                }
            },
            "required": ["entries"],
            "additionalProperties": False
        },
        "strict": True
    }
}

txt_file = Path("example/example0001-061.txt")

chunk = txt_file.read_text(encoding="utf-8")

response = client.chat.completions.create(
    model="gpt-4.1-mini-2025-04-14",
    temperature = 0.0,
    messages=[
        {"role": "system", "content": prompt},
        {"role": "user", "content": chunk}
    ],
    response_format=structured_output
)


In [ ]:
# Access the content string from the response object
json_string = response.choices[0].message.content
# Parse the string into a dictionary
data = json.loads(json_string) 
# Print with 'pretty' formatting
print(json.dumps(data, indent=4))

In [ ]:
# We recommend checking the full response object for additional metadata such as token counts etc.
print(response.model_dump_json(indent=4))